Leaf Classification

The purpose of this task is to classify leaf images using supervised machine learning. In the end the models are going to predict in which category does an unseen leaf belong to. In the beginning relevant features are extracted from the data like the glcm scores and different color channel value means and variances. These features form a feature vector that corresponds to each row of the original leaf data. This feature vector is used in training the different models to find similarities which indicate that a certain leaf belongs to a certain class.

The dataset consists of three leaf classes Basil, Lemon and Chinar. The original images contain RGB values which need to be converted to grayscale for the purpose of the analysis. The images are sourced from Kaggle.

To train the classifiers the following methods are used Ridge Classifier, Random Forest and Multi-Layer Perceptron.

# Leaf Classification

In [56]:
#All imports are done here

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage as ski

#For data-analysis methods
import sklearn as sk
import sklearn.preprocessing as skp
import sklearn.linear_model as sklm
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


#For future proof image handling instead of skimage.io
import imageio.v3 as iio

#For reading files from folders
from pathlib import Path

#Computer vision
import cv2 as cv




## Data Preparation

In [ ]:
#Import all images to their own list
basil = []
for file in Path('Basil').iterdir():
  if not file.is_file():
    continue

  basil.append(iio.imread(file))

chinar = []
for file in Path('Chinar').iterdir():
  if not file.is_file():
    continue

  chinar.append(iio.imread(file))

lemon = []
for file in Path('Lemon').iterdir():
  if not file.is_file():
    continue

  lemon.append(iio.imread(file))

In [ ]:
#Converting all images to 3-bit grayscale
#and saving them and each color channel to their own list
categories = [basil, chinar, lemon]
all_gray = []
all_red = []
all_green = []
all_blue = []

for leaves in categories:
  for leaf in leaves:
    grayLeaf = ski.color.rgb2gray(leaf)
    grayLeaf = np.uint8(np.clip(grayLeaf * 8, 0, 7))
    all_gray.append(grayLeaf)
    all_blue.append(leaf[:,:,0])
    all_green.append(leaf[:,:,1])
    all_red.append(leaf[:,:,2])

In [ ]:
#Creating a dataframe of with each picture and their label

#Images: 184, 120, 180
X = np.stack(all_gray)
X_width = X[:, ]

#Labels 0: basil, 1: chinar, 2: lemon
Y = np.concatenate([
    np.zeros(len(basil), dtype=int),
    np.ones(len(chinar), dtype=int),
    np.full(len(lemon), 2, dtype=int)
])

## Feature Extraction

### First order texture measures

In [60]:
#Mean values for each image and each RGB channel
#Result is the average value for a single image and a single color

all_red = np.array(all_red)
red_mean = np.mean(all_red, axis=(1,2))

all_green = np.array(all_green)
green_mean = np.mean(all_green, axis=(1,2))


all_blue = np.array(all_blue)
blue_mean = np.mean(all_blue, axis=(1,2))


In [61]:
#Variance values for each image and each RGB channel
#Result is the average value for a single image and a single color

red_var = np.var(all_red, axis=(1,2))
green_var = np.var(all_green, axis=(1,2))
blue_var = np.var(all_blue, axis=(1,2))

### Second order texture measures

In [62]:
#Gray-Level Co-Occurrence Matrix (GLCM)
#Correlation for GLCM means how similar each pixel is to it's neighbours.
#High correlation means there is less "texture" in the image so the whole image is more homogenous.

glcm_vertical_1 = []
glcm_vertical_2 = []
glcm_horizontal_1 = []
glcm_horizontal_2 = []

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[1], angles=[0], levels=8, symmetric=False, normed=True 
  )
  glcm_horizontal_1.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_horizontal_1 = np.array(glcm_horizontal_1)

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[2], angles=[0], levels=8, symmetric=False, normed=True 
  )
  glcm_horizontal_2.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_horizontal_2 = np.array(glcm_horizontal_2)

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[1], angles=[np.pi/2], levels=8, symmetric=False, normed=True 
  )
  glcm_vertical_1.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_vertical_1 = np.array(glcm_vertical_1)

for leaf in X:
  glcm = ski.feature.graycomatrix(
    leaf, distances=[2], angles=[np.pi/2], levels=8, symmetric=False, normed=True 
  )
  glcm_vertical_2.append(ski.feature.graycoprops(glcm, 'correlation')[0, 0])

glcm_vertical_2 = np.array(glcm_vertical_2)

In [ ]:
#Creation of the feature vector that combines all of the extracted features

feature_vector = pd.concat([
  pd.DataFrame(Y),
  pd.DataFrame(red_mean),
  pd.DataFrame(red_var),
  pd.DataFrame(green_mean),
  pd.DataFrame(green_var),
  pd.DataFrame(blue_mean),
  pd.DataFrame(blue_var),
  pd.DataFrame(glcm_vertical_1),
  pd.DataFrame(glcm_vertical_2),
  pd.DataFrame(glcm_horizontal_1),
  pd.DataFrame(glcm_horizontal_2),
], axis=1)

feature_vector = feature_vector.set_axis([
  'leaf'>
  'red_mean',
  'red_var',
  'green_mean', 
  'green_var', 
  'blue_mean', 
  'blue_var', 
  'glcm_vertical_1', 
  'glcm_vertical_2', 
  'glcm_horizontal_1', 
  'glcm_horizontal_2'
  ], axis=1)

## Additional image-based features

In [ ]:
#For each type of leaf finding the template leaf by the highest glcm_mean value

feature_vector['glcm_mean'] = feature_vector[[
  'glcm_vertical_1', 
  'glcm_vertical_2', 
  'glcm_horizontal_1', 
  'glcm_horizontal_2'
  ]].mean(axis=1)

idx = (feature_vector.groupby('leaf')['glcm_mean'].transform(max) == feature_vector['glcm_mean'])
template_leaves = feature_vector[idx]

Index([2, 86, 151], dtype='int64')


In [ ]:
#Comparing each leaf to each template leaf and adding the result to the feature vector

basil_scores = []
chinar_scores = []
lemon_scores = []

basil_template = all_gray[template_leaves['leaf'].index[0]]
chinar_template = all_gray[template_leaves['leaf'].index[1]]
lemon_template = all_gray[template_leaves['leaf'].index[2]]

for leaf in all_gray:
  res = cv.matchTemplate(leaf, basil_template, cv.TM_CCOEFF_NORMED)
  basil_scores.append(res.max())

for leaf in all_gray:
  res = cv.matchTemplate(leaf, chinar_template, cv.TM_CCOEFF_NORMED)
  chinar_scores.append(res.max())

for leaf in all_gray:
  res = cv.matchTemplate(leaf, lemon_template, cv.TM_CCOEFF_NORMED)
  lemon_scores.append(res.max())

template_matching_scores = pd.DataFrame([basil_scores, chinar_scores, lemon_scores])

template_matching_scores = template_matching_scores.T
template_matching_scores.columns = ['basil_match', 'chinar_match', 'lemon_match']

feature_vector = pd.concat([feature_vector, template_matching_scores], axis=1)

In [ ]:

y = feature_vector['leaf']
feature_vector_complete = feature_vector.drop(['leaf', 'glcm_mean'], axis=1)

scaler = skp.StandardScaler()

X = pd.DataFrame(scaler.fit_transform(feature_vector_complete))

(184, 13)
(184,)


In [67]:
def nested_cross_validation(model, params):
    NUM_TRIALS = 30

    nested_scores = np.zeros(NUM_TRIALS)

    # Loop for each trial
    for i in range(NUM_TRIALS):
        # Choose cross-validation techniques for the inner and outer loops,
        # independently of the dataset.
        # E.g "GroupKFold", "LeaveOneOut", "LeaveOneGroupOut", etc.
        inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=i)
        outer_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=i)

        # Nested CV with parameter optimization
        clf = GridSearchCV(estimator=model, param_grid=params, cv=inner_cv, n_jobs=-1, scoring='accuracy')
        nested_score = cross_val_score(clf, X=X, y=y, cv=outer_cv)
        nested_scores[i] = (nested_score.mean())

    return nested_scores

In [68]:
ridge_params = {'alpha': np.linspace(0,1,10), 'max_iter': np.arange(1,10)}
ridge_classifier = sklm.RidgeClassifier()
forest_params = {'n_estimators' : np.arange(10,100)}
forest_classifier = RandomForestClassifier()
mlp_params = {'hidden_layer_sizes': np.arange(1,10)}
mlp_classifier = MLPClassifier() 

ridge = nested_cross_validation(ridge_classifier, ridge_params)
print(ridge)
forest = nested_cross_validation(forest_classifier, forest_params)
print(forest)
mlp = nested_cross_validation(mlp_classifier, mlp_params)
print(mlp)

[0.94021739 0.92934783 0.94021739 0.92934783 0.94021739 0.93478261
 0.92391304 0.94021739 0.93478261 0.92391304 0.94021739 0.92934783
 0.94021739 0.92391304 0.92391304 0.92391304 0.91304348 0.93478261
 0.93478261 0.93478261 0.92934783 0.94021739 0.93478261 0.92391304
 0.93478261 0.93478261 0.93478261 0.94021739 0.91304348 0.94565217]
[0.89673913 0.91847826 0.89673913 0.92391304 0.9076087  0.89673913
 0.89130435 0.91304348 0.91304348 0.9076087  0.92391304 0.875
 0.89130435 0.875      0.91304348 0.91304348 0.89673913 0.91304348
 0.92391304 0.89673913 0.90217391 0.92934783 0.92391304 0.9076087
 0.88586957 0.9076087  0.92391304 0.91304348 0.91847826 0.91304348]


/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/n

[0.81521739 0.74456522 0.90217391 0.89673913 0.875      0.78804348
 0.8423913  0.89130435 0.90217391 0.81521739 0.80434783 0.84782609
 0.88586957 0.875      0.84782609 0.8423913  0.86413043 0.76086957
 0.86413043 0.85869565 0.88043478 0.88586957 0.91847826 0.86956522
 0.88043478 0.89130435 0.74456522 0.75543478 0.91847826 0.88586957]


/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/asteer/data/lib/python3.12/site-packages/sklearn/n

In [69]:
print(ridge.mean())
print(forest.mean())
print(mlp.mean())

0.9322463768115943
0.9070652173913045
0.8518115942028985
